# ARIMA & SARIMA — Time Series Forecasting

---

## Building Blocks

### AR — AutoRegressive model (order $p$)

The current value is a linear combination of the $p$ previous values:

$$X_t = c + \sum_{i=1}^p \phi_i X_{t-i} + \varepsilon_t$$

### MA — Moving Average model (order $q$)

The current value depends on $q$ previous **error terms**:

$$X_t = \mu + \varepsilon_t + \sum_{j=1}^q \theta_j \varepsilon_{t-j}$$

### ARIMA$(p, d, q)$

Combines AR + MA with $d$ rounds of **differencing** to achieve stationarity.

### SARIMA$(p, d, q)(P, D, Q)_m$

Adds seasonal AR, MA and differencing terms at lag $m$.

---

## Workflow

```
Raw series → Stationarity test (ADF) → Differencing → ACF/PACF → Order selection → Fit → Diagnostics → Forecast
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)
print('Imports OK')

---
## 1  Dataset — Air Passengers (classic seasonal benchmark)

In [ ]:
# Load AirPassengers from statsmodels
from statsmodels.datasets import get_rdataset
ap = get_rdataset('AirPassengers').data
ap['date'] = pd.date_range('1949-01', periods=len(ap), freq='MS')
ap = ap.set_index('date')['value']
ap.name = 'Passengers'

print(ap.describe())

plt.figure(figsize=(11, 4))
ap.plot(lw=2)
plt.title('Monthly Air Passengers (1949–1960)')
plt.ylabel('Passengers (thousands)')
plt.tight_layout()
plt.show()

---
## 2  Stationarity — Augmented Dickey-Fuller Test

In [ ]:
def adf_summary(series, name=''):
    result = adfuller(series.dropna())
    is_stationary = result[1] < 0.05
    print(f'\n{name}')
    print(f'  ADF stat : {result[0]:.4f}')
    print(f'  p-value  : {result[1]:.4f}  → {"STATIONARY" if is_stationary else "NON-STATIONARY"}')
    return is_stationary


adf_summary(ap, 'Raw series')

# Log-transform to stabilise variance (multiplicative seasonality)
ap_log = np.log(ap)
adf_summary(ap_log, 'Log-transformed')

# Seasonal differencing (lag 12) then first-order diff
ap_log_d12 = ap_log.diff(12)
ap_log_d12_d1 = ap_log_d12.diff(1)
adf_summary(ap_log_d12_d1.dropna(), 'Log + seasonal diff(12) + diff(1)')

---
## 3  ACF & PACF — Order Selection

In [ ]:
diff_series = ap_log_d12_d1.dropna()

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
plot_acf(diff_series, lags=36, ax=axes[0], title='ACF — differenced log passengers')
plot_pacf(diff_series, lags=36, ax=axes[1], title='PACF — differenced log passengers', method='ywm')
plt.tight_layout()
plt.show()

print("""
Reading the plots:
  ACF  — significant spike at lag 1 → MA(1);  spike at lag 12 → SMA(1)
  PACF — significant spike at lag 1 → AR(1);  spike at lag 12 → SAR(1)
  Suggests: SARIMA(1,1,1)(1,1,1)_12
""")

---
## 4  ARIMA — Non-Seasonal Baseline

In [ ]:
# Train/test split — hold out last 24 months
HOLDOUT = 24
train_log = ap_log.iloc[:-HOLDOUT]
test_log  = ap_log.iloc[-HOLDOUT:]

arima = ARIMA(train_log, order=(2, 1, 2))
arima_fit = arima.fit()

forecast_arima = arima_fit.forecast(steps=HOLDOUT)
forecast_arima_orig = np.exp(forecast_arima)
test_orig = np.exp(test_log)

mae_arima = mean_absolute_error(test_orig, forecast_arima_orig)
rmse_arima = np.sqrt(mean_squared_error(test_orig, forecast_arima_orig))
print(f'ARIMA(2,1,2) — MAE: {mae_arima:.2f}  RMSE: {rmse_arima:.2f}')

---
## 5  SARIMA — Seasonal Model

In [ ]:
sarima = SARIMAX(train_log,
                 order=(1, 1, 1),
                 seasonal_order=(1, 1, 1, 12),
                 enforce_stationarity=False,
                 enforce_invertibility=False)
sarima_fit = sarima.fit(disp=False)
print(sarima_fit.summary())

In [ ]:
forecast_sarima = sarima_fit.forecast(steps=HOLDOUT)
forecast_sarima_orig = np.exp(forecast_sarima)

mae_sarima = mean_absolute_error(test_orig, forecast_sarima_orig)
rmse_sarima = np.sqrt(mean_squared_error(test_orig, forecast_sarima_orig))
print(f'SARIMA(1,1,1)(1,1,1)_12 — MAE: {mae_sarima:.2f}  RMSE: {rmse_sarima:.2f}')

plt.figure(figsize=(12, 5))
np.exp(ap_log).plot(label='Actual', lw=2)
forecast_arima_orig.plot(ls='--', label=f'ARIMA (RMSE={rmse_arima:.1f})', lw=2)
forecast_sarima_orig.plot(ls='--', label=f'SARIMA (RMSE={rmse_sarima:.1f})', lw=2)
plt.axvline(test_orig.index[0], color='black', ls=':', lw=2, label='Forecast start')
plt.title('ARIMA vs SARIMA — Air Passengers Forecast')
plt.legend()
plt.tight_layout()
plt.show()

---
## 6  Residual Diagnostics

In [ ]:
sarima_fit.plot_diagnostics(figsize=(13, 8))
plt.suptitle('SARIMA Residual Diagnostics', fontsize=13)
plt.tight_layout()
plt.show()

print("""
Ideal diagnostics:
  Standardised residuals : random, no pattern
  Histogram + KDE        : close to N(0,1)
  Normal Q-Q             : points on diagonal line
  Correlogram (ACF)      : all lags within confidence bands
""")

---
## 7  Walk-Forward Validation

In [ ]:
# Rolling 1-step-ahead forecast — the most realistic evaluation
history = list(ap_log.iloc[:-HOLDOUT])
preds_wf = []

for t in range(HOLDOUT):
    model = SARIMAX(history,
                    order=(1, 1, 1),
                    seasonal_order=(1, 1, 1, 12),
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    fit = model.fit(disp=False)
    yhat = fit.forecast(steps=1)[0]
    preds_wf.append(yhat)
    history.append(float(ap_log.iloc[-HOLDOUT + t]))  # true value added

wf_orig = np.exp(np.array(preds_wf))
mae_wf  = mean_absolute_error(test_orig, wf_orig)
rmse_wf = np.sqrt(mean_squared_error(test_orig, wf_orig))
print(f'Walk-forward SARIMA — MAE: {mae_wf:.2f}  RMSE: {rmse_wf:.2f}')

plt.figure(figsize=(12, 4))
test_orig.plot(label='Actual', lw=2)
pd.Series(wf_orig, index=test_orig.index).plot(ls='--', lw=2, label=f'Walk-forward SARIMA (RMSE={rmse_wf:.1f})')
plt.title('Walk-Forward Validation — SARIMA 1-step-ahead')
plt.legend()
plt.tight_layout()
plt.show()

---
## 8  Model Comparison Summary

In [ ]:
results = pd.DataFrame([
    {'Model': 'ARIMA(2,1,2)',               'MAE': mae_arima,  'RMSE': rmse_arima},
    {'Model': 'SARIMA(1,1,1)(1,1,1)_12',    'MAE': mae_sarima, 'RMSE': rmse_sarima},
    {'Model': 'Walk-forward SARIMA',         'MAE': mae_wf,     'RMSE': rmse_wf},
]).set_index('Model').round(2)

print(results)

results['RMSE'].plot(kind='bar', figsize=(8, 4), color=['#e41a1c', '#377eb8', '#4daf4a'])
plt.ylabel('RMSE (passengers)')  
plt.title('Forecast RMSE Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

---
## Summary

| Model | When to use |
|---|---|
| **AR(p)** | Series depends on own lags |
| **MA(q)** | Series depends on past errors |
| **ARIMA(p,d,q)** | Non-seasonal, non-stationary data |
| **SARIMA(p,d,q)(P,D,Q)_m** | Seasonal patterns (monthly, quarterly) |

**Order selection cheatsheet:**
- $d$ → ADF test + differencing
- $p$ → PACF cut-off
- $q$ → ACF cut-off
- $D$ → seasonal unit root test
- Confirm with AIC/BIC (lower is better)

**See also:** Topic 116 (advanced time series ML), Topic 108 (regression evaluation metrics)